# Backtest Comparison

Run the same strategy across multiple tickers and compare results side-by-side.

**Sections**
1. Setup
2. Multi-ticker backtest
3. Results table
4. Equity curves
5. Risk-adjusted comparison
6. Trade statistics heatmap

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.automated_trading_system import AutomatedTradingSystem

TICKERS    = ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'NVDA']
START_DATE = '2023-01-01'
END_DATE   = '2024-01-01'
CAPITAL    = 10_000

os.makedirs('../runs', exist_ok=True)
print('Tickers:', TICKERS)
print('Period :', START_DATE, '->', END_DATE, '| Capital: $', CAPITAL)

## 2. Run Backtests

In [ ]:
systems = {}
results = {}

for ticker in TICKERS:
    print('  Backtesting', ticker, '...', end=' ')
    s = AutomatedTradingSystem(
        initial_capital=CAPITAL, ticker=ticker,
        max_positions=5, max_position_size_pct=0.05,
    )
    r = s.backtest(start_date=START_DATE, end_date=END_DATE)
    systems[ticker] = s
    results[ticker] = r
    print('return={:+.1f}%  win_rate={:.0f}%'.format(
        r['portfolio']['return_pct'], r['trades']['win_rate']))

print('Done.')

## 3. Results Table

In [ ]:
rows = []
for ticker, r in results.items():
    p = r['portfolio']
    t = r['trades']
    closed = systems[ticker].portfolio.closed_positions

    equity, peak, max_dd = CAPITAL, CAPITAL, 0.0
    for tr in sorted(closed, key=lambda x: x.exit_date or x.entry_date):
        equity += tr.profit_loss
        if equity > peak: peak = equity
        dd = (equity - peak) / peak * 100
        if dd < max_dd: max_dd = dd

    rows.append({
        'Ticker'        : ticker,
        'Return (%)'    : round(p['return_pct'], 2),
        'Final Value'   : round(p['total_value'], 2),
        'Total Trades'  : t['total_trades'],
        'Win Rate (%)'  : round(t['win_rate'], 1),
        'Profit Factor' : round(t.get('profit_factor', 0), 2),
        'Max DD (%)'    : round(max_dd, 2),
        'Signals'       : r['signals']['total_signals'],
    })

cmp_df = pd.DataFrame(rows).set_index('Ticker')
print(cmp_df.to_string())

In [ ]:
# Colour-coded table (green=best, red=worst per column)
def highlight(df):
    styled = df.style
    for col in ['Return (%)', 'Win Rate (%)', 'Profit Factor']:
        if col in df.columns:
            styled = styled.highlight_max(subset=[col], color='#c6efce')
            styled = styled.highlight_min(subset=[col], color='#ffc7ce')
    if 'Max DD (%)' in df.columns:
        styled = styled.highlight_min(subset=['Max DD (%)'], color='#c6efce')
        styled = styled.highlight_max(subset=['Max DD (%)'], color='#ffc7ce')
    return styled

try:
    display(highlight(cmp_df))
except NameError:
    print(cmp_df.to_string())

## 4. Equity Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Equity Curves  |  {} -> {}'.format(START_DATE, END_DATE),
             fontsize=13, fontweight='bold')

colors = plt.cm.tab10.colors

for i, ticker in enumerate(TICKERS):
    closed = systems[ticker].portfolio.closed_positions
    if not closed:
        continue
    eq_dates, eq_vals, running = [], [], CAPITAL
    for tr in sorted(closed, key=lambda x: x.exit_date or x.entry_date):
        running += tr.profit_loss
        eq_dates.append(tr.exit_date)
        eq_vals.append(running)

    axes[0].plot(eq_dates, eq_vals, color=colors[i], linewidth=1.5, label=ticker)
    pct = [(v / CAPITAL - 1) * 100 for v in eq_vals]
    axes[1].plot(eq_dates, pct, color=colors[i], linewidth=1.5, label=ticker)

axes[0].axhline(CAPITAL, color='grey', linewidth=0.8, linestyle='--', label='Initial')
axes[0].set_title('Portfolio Value ($)'); axes[0].set_ylabel('Value ($)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].axhline(0, color='grey', linewidth=0.8, linestyle='--')
axes[1].set_title('Return (%)'); axes[1].set_ylabel('Return (%)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../runs/nb_equity_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: runs/nb_equity_curves.png')

## 5. Risk-Adjusted Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Risk-Adjusted Metrics', fontsize=12, fontweight='bold')

tickers_list = list(cmp_df.index)
x = np.arange(len(tickers_list))
bar_colors = ['#2ca02c' if v >= 0 else '#d62728' for v in cmp_df['Return (%)']]

for ax, col, label, color, ref in [
    (axes[0], 'Return (%)',    'Return (%)',    bar_colors, 0),
    (axes[1], 'Win Rate (%)',  'Win Rate (%)',  'steelblue', 50),
    (axes[2], 'Profit Factor', 'Profit Factor', 'darkorange', 1.0),
]:
    c = color if isinstance(color, list) else color
    ax.bar(x, cmp_df[col], color=c, alpha=0.85)
    ax.axhline(ref, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels(tickers_list)
    ax.set_title(label); ax.set_ylabel(label); ax.grid(axis='y', alpha=0.3)
    for j, v in enumerate(cmp_df[col]):
        ax.text(j, v + abs(v) * 0.03 if v >= 0 else v - abs(v) * 0.06,
                '{:.2f}'.format(v), ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('../runs/nb_risk_adjusted.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Trade Statistics Heatmap

In [ ]:
metrics  = ['Return (%)', 'Win Rate (%)', 'Profit Factor', 'Total Trades', 'Max DD (%)']
heat_df  = cmp_df[metrics].astype(float)
norm_df  = heat_df.copy()

for col in metrics:
    lo, hi = heat_df[col].min(), heat_df[col].max()
    rng = hi - lo
    norm_df[col] = (heat_df[col] - lo) / rng if rng > 0 else 0.5
norm_df['Max DD (%)'] = 1 - norm_df['Max DD (%)']  # lower drawdown is better

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(norm_df.values.T, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(heat_df.index)));  ax.set_xticklabels(heat_df.index, fontsize=10)
ax.set_yticks(range(len(metrics)));        ax.set_yticklabels(metrics, fontsize=9)

for (r, c), val in np.ndenumerate(heat_df.values.T):
    ax.text(c, r, '{:.1f}'.format(val), ha='center', va='center',
            fontsize=9, fontweight='bold')

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04, label='Relative score (green=best)')
ax.set_title('Strategy Comparison Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../runs/nb_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: runs/nb_heatmap.png')

## Summary

In [ ]:
col_ret = 'Return (%)'
col_wr  = 'Win Rate (%)'
best    = cmp_df[col_ret].idxmax()
worst   = cmp_df[col_ret].idxmin()

print('Best return  :', best,  '({:+.2f}%)'.format(cmp_df.loc[best,  col_ret]))
print('Worst return :', worst, '({:+.2f}%)'.format(cmp_df.loc[worst, col_ret]))
print('Avg return   : {:+.2f}%'.format(cmp_df[col_ret].mean()))
print('Avg win rate : {:.1f}%'.format(cmp_df[col_wr].mean()))